In [1]:
import pandas as pd

# temporary formatting purposes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [2]:
data = pd.read_csv("professors_info.csv")
data.head()

,SID,School Name,Professor Name,Overall Quality,Num Reviews,Department,Class Taken,Review Date,Quality,Difficulty,Review Text
0,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"Sep 6th, 2024",5.0,2.0,"Best Professor I've Ever Had. He explains complex concepts with such clarity and depth that everything feels understandable. What truly sets him apart is how deeply he cares about his students. His kindness and compassion create an incredibly supportive learning environment. On top of that, he's hilarious, making class not only informative but fun!"
1,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"Jun 14th, 2024",5.0,2.0,Probably one of the best math professors at wash
2,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH310W,"Jun 4th, 2024",5.0,2.0,"Schaefer is a very skilled educator and he is very qualified to teach the sensitive 310W. Schaefer's 310 included a good coverage of foundational topics. The class is very difficult, that's just the nature of it. Schaefer doesn't seem to hold punches but still gives exceptional lectures and is very accessible and helpful outside of class."
3,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"Jun 4th, 2024",5.0,2.0,Schaefer remains unparalleled for Calculus lectures. Precup was also very good.
4,1147,Washington University in St. Louis,Karl Schaefer,4.8,20,Mathematics,MATH233,"May 22nd, 2024",5.0,2.0,Professor Schaefer is an amazing Professor. He is very organized and gives practice tests and quizzes to help people succeed.


<h1>Data Pre-Processing</h1>

Convert all course codes to uppercase

In [5]:
data['Class Taken'] = data['Class Taken'].str.upper()

If the course code ends with a letter, remove it.
<br>
Example: CHEM111A --> Chem111

In [7]:
data['Class Taken'] = data['Class Taken'].str.replace(r'[A-Z]$', '', regex=True)

Remove rows where the class taken is not given (empty Class Taken)
<br>
There is only one empty course title, so this is feasible

In [9]:
data = data[data['Class Taken'].str.strip() != '']

Let's use fuzzy matching NLP to group course codes within each department that share at least a 92% similarity. <br>
This will standardize course codes to a single common course code, if they're similar enough

Example:<br>
CHEM111 and CHM111 --> CHEM111

In [11]:
from fuzzywuzzy import process

In [12]:
def similar_course_mapper(department):
    unique_courses = department['Class Taken'].unique()
    course_map = {}
    
    for course in unique_courses:
        # find the closest matches to each course within the department
        matches = process.extract(course, unique_courses)
        # group courses together if they have a similarity score >= 92/100%
        similar_courses = [match for match, score in matches if score >= 92]
        
        if similar_courses:
            # if a similar course grouping exists --> the new course code will be the most popular course code of that group
            most_frequent_course = department[department['Class Taken'].isin(similar_courses)]['Class Taken'].mode()
            if len(most_frequent_course) > 0:
                most_frequent_course_value = most_frequent_course[0]
                for similar_course in similar_courses:
                    course_map[similar_course] = most_frequent_course_value
                    
    return course_map

In [13]:
# apply the function above to get the correct course code mappings within each department
course_corrections = data.groupby('Department').apply(similar_course_mapper, include_groups=False).to_dict()

In [14]:
# get the corrected course code for each row based on the department and class taken, otherwise default original course code
def correct_courses(row, corrections):
    return corrections.get(row['Department'], {}).get(row['Class Taken'], row['Class Taken'])

In [15]:
# apply the function above to correct the course codes
data['Class Taken Corrected'] = data.apply(correct_courses, corrections=course_corrections, axis=1)

Only allow keep course codes that match the following format: [A-Z]+[0-9]{3,4}[A-Z]?

In other words, only keep course codes that start with alphabetical letters, followed by 3-4 digits, and an optional alphabetical letter indicating a type of course (i.e. T, M, A, S, D)

Examples:
Accepted Course Codes: CSE330S, CSE217A, CSE240
Not Accepted Course Codes: 100B, 100, B

In [17]:
course_code_pattern = r'^[A-Z]+[0-9]{3,4}[A-Z]?$'

print(data.shape)
data = data[data['Class Taken'].str.contains(course_code_pattern, regex=True)]
print(data.shape)

(6443, 12)
(4857, 12)


In [18]:
data[["Class Taken", "Class Taken Corrected"]]

,Class Taken,Class Taken Corrected
0,MATH233,MATH233
1,MATH233,MATH233
2,MATH310,MATH310
3,MATH233,MATH233
4,MATH233,MATH233
5,MATH233,MATH233
6,MATH233,MATH233
7,MATH233,MATH233
8,MATH233,MATH233
9,MATH233,MATH233
